## Training: BERT + ViT (late fusion)

This notebook trains a multimodal classifier on the Fakeddit subset created by `dataset_downloader.ipynb`.

### Inputs
- `CLEANDFPATH`: CSV containing at least `id`, a text column (e.g. `clean_title`), and label column (e.g. `6_way_label`)
- `IMAGEDIR`: folder of cached images named `{id}.jpg`

### Model
- Text: `bert-base-uncased` (CLS embedding)
- Image: `google/vit-base-patch16-224-in21k` (CLS embedding)
- Fusion: project each to 512-d → concatenate → MLP classifier

### Outputs (saved to `SAVE_DIR`)
- Best checkpoint (`BEST_MODEL_PATH`) and final checkpoint (`FINAL_MODEL_PATH`)
- Train/val/test CSV splits
- Training history + config JSON
- Validation/test predictions + test report

### Notes
- This uses **class-weighted cross entropy** and a simple early-stopping rule.
- The backbones are partially frozen at the start to stabilize training on a smaller dataset.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
!pip -q install -U transformers accelerate scikit-learn

In [ ]:
import os
import json
import time
import copy
import math
import random
import shutil
import numpy as np
import pandas as pd

from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
import torch.optim.lr_scheduler as lr_scheduler
from torch.cuda.amp import autocast, GradScaler

from torchvision import transforms

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support
)

from transformers import (
    BertTokenizer,
    BertModel,
    AutoImageProcessor,
    ViTModel
)

In [ ]:
SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

torch.backends.cudnn.benchmark = True

## 1) Setup + paths

Mount Drive, install dependencies, set a random seed, and define the dataset/model/training hyperparameters and output paths.


In [ ]:
CLEANDFPATH = "/content/drive/MyDrive/Fake-news-detector/multimodal_only_samples/working/clean_df.csv"
IMAGEDIR = "/content/drive/MyDrive/Fake-news-detector/multimodal_only_samples/working/images"

SAVE_DIR = "/content/drive/MyDrive/Fake-news-detector/multimodal_only_samples/bert_vit_v2_2"
os.makedirs(SAVE_DIR, exist_ok=True)

EPOCH_CKPT_DIR = os.path.join(SAVE_DIR, "epoch_checkpoints")
os.makedirs(EPOCH_CKPT_DIR, exist_ok=True)

BEST_MODEL_PATH = os.path.join(SAVE_DIR, "bert_vit_v2_2_best_score.pth")
FINAL_MODEL_PATH = os.path.join(SAVE_DIR, "bert_vit_v2_2_final.pth")
CONFIG_PATH = os.path.join(SAVE_DIR, "bert_vit_v2_2_config.json")
TEST_PRED_PATH = os.path.join(SAVE_DIR, "bert_vit_v2_2_test_predictions.csv")
VAL_PRED_PATH = os.path.join(SAVE_DIR, "bert_vit_v2_2_val_predictions.csv")
REPORT_PATH = os.path.join(SAVE_DIR, "bert_vit_v2_2_test_report.txt")
CLASS_WEIGHTS_PATH = os.path.join(SAVE_DIR, "bert_vit_v2_2_class_weights.json")
HISTORY_PATH = os.path.join(SAVE_DIR, "bert_vit_v2_2_history.json")
TRAIN_SPLIT_PATH = os.path.join(SAVE_DIR, "train_split_v2_2.csv")
VAL_SPLIT_PATH = os.path.join(SAVE_DIR, "val_split_v2_2.csv")
TEST_SPLIT_PATH = os.path.join(SAVE_DIR, "test_split_v2_2.csv")

BERT_NAME = "bert-base-uncased"
VIT_NAME = "google/vit-base-patch16-224-in21k"

MAX_LEN = 80
BATCH_SIZE = 16
NUM_EPOCHS = 12
HEAD_LR = 1e-4
BACKBONE_LR = 5e-6
WEIGHT_DECAY = 1e-4
DROPOUT = 0.4
LABEL_SMOOTHING = 0.02
EARLY_STOPPING_PATIENCE = 4
EARLY_STOPPING_DELTA = 0.001

NUM_WORKERS = 4
ACTIVE_IMAGE_DIR = IMAGEDIR

print("CLEANDFPATH exists:", os.path.exists(CLEANDFPATH))
print("IMAGEDIR exists:", os.path.exists(IMAGEDIR))
print("ACTIVE_IMAGE_DIR:", ACTIVE_IMAGE_DIR)
print("SAVE_DIR:", SAVE_DIR)

In [ ]:
df = pd.read_csv(CLEANDFPATH)
print("Shape:", df.shape)
print("Columns:", list(df.columns))
df.head(3)

In [ ]:
def pick_first_existing(columns, candidates):
    for c in candidates:
        if c in columns:
            return c
    return None

TEXT_COL = pick_first_existing(df.columns, ["clean_title", "cleantitle", "title"])
LABEL_COL = pick_first_existing(df.columns, ["6_way_label", "6waylabel", "label"])
ID_COL = pick_first_existing(df.columns, ["id"])

print("TEXT_COL =", TEXT_COL)
print("LABEL_COL =", LABEL_COL)
print("ID_COL =", ID_COL)

assert TEXT_COL is not None, "No text column found."
assert LABEL_COL is not None, "No label column found."
assert ID_COL is not None, "No id column found."

In [ ]:
df = df.copy()
df[TEXT_COL] = df[TEXT_COL].astype(str).fillna("")
df[LABEL_COL] = df[LABEL_COL].astype(int)
df[ID_COL] = df[ID_COL].astype(str)

all_files = set(os.listdir(IMAGEDIR))
df = df[df[ID_COL].apply(lambda x: f"{x}.jpg" in all_files)].reset_index(drop=True)

print("Filtered shape (existing images only):", df.shape)
print(df[LABEL_COL].value_counts().sort_index())

## 2) Load `clean_df.csv` and create train/val/test splits

We:
- infer which columns contain text/label/id (to tolerate slightly different CSV schemas)
- filter out rows whose images are missing from `IMAGEDIR`
- create stratified train/val/test splits and save them for reproducibility


In [ ]:
RANDOM_STATE = 42

df_train, df_test = train_test_split(
    df,
    test_size=0.2,
    stratify=df[LABEL_COL],
    random_state=RANDOM_STATE
)

df_test, df_val = train_test_split(
    df_test,
    test_size=0.5,
    stratify=df_test[LABEL_COL],
    random_state=RANDOM_STATE
)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

df_train.to_csv(TRAIN_SPLIT_PATH, index=False)
df_val.to_csv(VAL_SPLIT_PATH, index=False)
df_test.to_csv(TEST_SPLIT_PATH, index=False)

print("Train:", df_train.shape)
print("Val:", df_val.shape)
print("Test:", df_test.shape)
print("\nTrain class counts:\n", df_train[LABEL_COL].value_counts().sort_index())

In [ ]:
bert_tokenizer = BertTokenizer.from_pretrained(BERT_NAME)
vit_processor = AutoImageProcessor.from_pretrained(VIT_NAME)

size_cfg = vit_processor.size
if isinstance(size_cfg, dict):
    if "height" in size_cfg and "width" in size_cfg:
        IMG_H = size_cfg["height"]
        IMG_W = size_cfg["width"]
    elif "shortest_edge" in size_cfg:
        IMG_H = size_cfg["shortest_edge"]
        IMG_W = size_cfg["shortest_edge"]
    else:
        IMG_H = 224
        IMG_W = 224
else:
    IMG_H = 224
    IMG_W = 224

IMAGE_MEAN = vit_processor.image_mean
IMAGE_STD = vit_processor.image_std

image_transform = transforms.Compose([
    transforms.Resize((IMG_H, IMG_W)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGE_MEAN, std=IMAGE_STD)
])

NUM_CLASSES = len(sorted(df[LABEL_COL].unique()))

print("IMG_H, IMG_W:", IMG_H, IMG_W)
print("IMAGE_MEAN:", IMAGE_MEAN)
print("IMAGE_STD:", IMAGE_STD)
print("NUM_CLASSES:", NUM_CLASSES)

In [ ]:
train_text_enc = bert_tokenizer(
    df_train[TEXT_COL].astype(str).tolist(),
    add_special_tokens=True,
    max_length=MAX_LEN,
    truncation=True,
    padding="max_length",
    return_attention_mask=True
)

val_text_enc = bert_tokenizer(
    df_val[TEXT_COL].astype(str).tolist(),
    add_special_tokens=True,
    max_length=MAX_LEN,
    truncation=True,
    padding="max_length",
    return_attention_mask=True
)

test_text_enc = bert_tokenizer(
    df_test[TEXT_COL].astype(str).tolist(),
    add_special_tokens=True,
    max_length=MAX_LEN,
    truncation=True,
    padding="max_length",
    return_attention_mask=True
)

print("Pretokenization complete.")
print("Train encoded rows:", len(train_text_enc["input_ids"]))
print("Val encoded rows:", len(val_text_enc["input_ids"]))
print("Test encoded rows:", len(test_text_enc["input_ids"]))

In [ ]:
class FakedditBERTViTFastDataset(Dataset):
    def __init__(
        self,
        df,
        image_dir,
        image_transform,
        text_encodings,
        text_col,
        label_col,
        id_col
    ):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.image_transform = image_transform
        self.text_encodings = text_encodings
        self.text_col = text_col
        self.label_col = label_col
        self.id_col = id_col

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]

        label = int(row[self.label_col])
        image_id = str(row[self.id_col])
        text = str(row[self.text_col])

        img_path = os.path.join(self.image_dir, f"{image_id}.jpg")
        image = Image.open(img_path).convert("RGB")
        pixel_values = self.image_transform(image)

        return {
            "input_ids": torch.tensor(self.text_encodings["input_ids"][index], dtype=torch.long),
            "attention_mask": torch.tensor(self.text_encodings["attention_mask"][index], dtype=torch.long),
            "pixel_values": pixel_values,
            "labels": torch.tensor(label, dtype=torch.long),
            "image_id": image_id,
            "text": text
        }

In [ ]:
train_dataset = FakedditBERTViTFastDataset(
    df_train, ACTIVE_IMAGE_DIR, image_transform, train_text_enc, TEXT_COL, LABEL_COL, ID_COL
)

val_dataset = FakedditBERTViTFastDataset(
    df_val, ACTIVE_IMAGE_DIR, image_transform, val_text_enc, TEXT_COL, LABEL_COL, ID_COL
)

test_dataset = FakedditBERTViTFastDataset(
    df_test, ACTIVE_IMAGE_DIR, image_transform, test_text_enc, TEXT_COL, LABEL_COL, ID_COL
)

loader_kwargs = dict(
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True if NUM_WORKERS > 0 else False
)

train_loader = DataLoader(
    train_dataset,
    shuffle=True,
    **loader_kwargs
)

val_loader = DataLoader(
    val_dataset,
    shuffle=False,
    **loader_kwargs
)

test_loader = DataLoader(
    test_dataset,
    shuffle=False,
    **loader_kwargs
)

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))

sample = next(iter(train_loader))
print("input_ids:", sample["input_ids"].shape)
print("attention_mask:", sample["attention_mask"].shape)
print("pixel_values:", sample["pixel_values"].shape)
print("labels:", sample["labels"].shape)

## 3) Tokenization + image preprocessing

Text is tokenized once (BERT tokenizer) and stored in memory to speed up training.
Images are resized/normalized using the ViT processor’s recommended mean/std.


## 4) Dataset, model, and training utilities

This section defines:
- the PyTorch `Dataset` that pairs pre-tokenized text with images
- the BERT+ViT fusion model
- freezing strategy, early stopping, and the train/eval loops


In [ ]:
class BERTViTClassifierV22(nn.Module):
    def __init__(
        self,
        num_classes=6,
        bert_name="bert-base-uncased",
        vit_name="google/vit-base-patch16-224-in21k",
        dropout=0.4
    ):
        super().__init__()

        self.text_model = BertModel.from_pretrained(bert_name)
        self.image_model = ViTModel.from_pretrained(vit_name)

        self.text_proj = nn.Sequential(
            nn.Linear(self.text_model.config.hidden_size, 512),
            nn.GELU(),
            nn.LayerNorm(512),
            nn.Dropout(dropout)
        )

        self.image_proj = nn.Sequential(
            nn.Linear(self.image_model.config.hidden_size, 512),
            nn.GELU(),
            nn.LayerNorm(512),
            nn.Dropout(dropout)
        )

        self.classifier = nn.Sequential(
            nn.Linear(1024, 512),
            nn.GELU(),
            nn.LayerNorm(512),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes)
        )

    def forward(self, input_ids, attention_mask, pixel_values):
        text_outputs = self.text_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        text_feat = text_outputs.last_hidden_state[:, 0, :]

        image_outputs = self.image_model(
            pixel_values=pixel_values
        )
        image_feat = image_outputs.last_hidden_state[:, 0, :]

        text_feat = self.text_proj(text_feat)
        image_feat = self.image_proj(image_feat)

        fused = torch.cat([text_feat, image_feat], dim=1)
        logits = self.classifier(fused)
        return logits

In [ ]:
def freeze_backbones(model):
    """Freeze the lower layers of BERT and ViT.

    Rationale: on smaller datasets, fully fine-tuning both backbones from the start can
    overfit and/or destabilize training. We freeze embeddings + early encoder blocks and
    train the projection + classifier layers (and any unfrozen top backbone layers).
    """
    # Freeze BERT embeddings and early encoder blocks
    for param in model.text_model.embeddings.parameters():
        param.requires_grad = False
    for layer in model.text_model.encoder.layer[:8]:
        for param in layer.parameters():
            param.requires_grad = False

    # Freeze ViT embeddings and early encoder blocks
    for param in model.image_model.embeddings.parameters():
        param.requires_grad = False
    for layer in model.image_model.encoder.layer[:8]:
        for param in layer.parameters():
            param.requires_grad = False


def print_trainable_parameters(model):
    """Quick sanity check: how many parameters will update?"""
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable params: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")

In [ ]:
model = BERTViTClassifierV22(
    num_classes=NUM_CLASSES,
    bert_name=BERT_NAME,
    vit_name=VIT_NAME,
    dropout=DROPOUT
).to(device)

freeze_backbones(model)
print_trainable_parameters(model)

labels = df_train[LABEL_COL].to_numpy()
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(labels),
    y=labels
)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

with open(CLASS_WEIGHTS_PATH, "w") as f:
    json.dump({str(i): float(w) for i, w in enumerate(class_weights.detach().cpu().tolist())}, f, indent=2)

criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=LABEL_SMOOTHING
)

head_params = []
backbone_params = []

for name, param in model.named_parameters():
    if not param.requires_grad:
        continue
    if any(k in name for k in ["text_proj", "image_proj", "classifier"]):
        head_params.append(param)
    else:
        backbone_params.append(param)

optimizer = AdamW(
    [
        {"params": head_params, "lr": HEAD_LR, "weight_decay": WEIGHT_DECAY},
        {"params": backbone_params, "lr": BACKBONE_LR, "weight_decay": WEIGHT_DECAY}
    ]
)

scheduler = lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=1,
    min_lr=1e-6
)

scaler = GradScaler(enabled=(device == "cuda"))

history = {
    "train_loss": [],
    "val_loss": [],
    "val_acc": [],
    "val_precision_weighted": [],
    "val_recall_weighted": [],
    "val_f1_weighted": [],
    "val_f1_macro": [],
    "val_score": [],
    "val_cls3_recall": [],
    "val_cls3_f1": [],
    "val_cls4_recall": [],
    "val_cls4_f1": []
}

print("Class weights:", class_weights)

In [ ]:
class EarlyStoppingByScore:
    def __init__(self, patience=4, delta=0.001, verbose=True):
        self.patience = patience
        self.delta = delta
        self.verbose = verbose
        self.best_score = None
        self.counter = 0
        self.early_stop = False

    def step(self, score):
        improved = False

        if self.best_score is None or score > self.best_score + self.delta:
            self.best_score = score
            self.counter = 0
            improved = True
            if self.verbose:
                print(f"New best score: {score:.4f}")
        else:
            self.counter += 1
            if self.verbose:
                print(f"EarlyStopping counter: {self.counter} out of {self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True

        return improved

def compute_selection_score(weighted_f1, macro_f1):
    """Single scalar used for checkpointing + LR scheduling.

    Weighted-F1 tracks overall performance (accounts for class imbalance), while macro-F1
    prevents the model from ignoring minority classes.
    """
    return 0.7 * weighted_f1 + 0.3 * macro_f1

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, scaler, device, epoch_num=None, print_every=80):
    model.train()
    running_loss = 0.0
    start = time.time()

    for batch_idx, batch in enumerate(loader, start=1):
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        pixel_values = batch["pixel_values"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast(enabled=(device == "cuda")):
            logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                pixel_values=pixel_values
            )
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * labels.size(0)

        if batch_idx % print_every == 0 or batch_idx == 1 or batch_idx == len(loader):
            elapsed = (time.time() - start) / 60
            avg_loss_so_far = running_loss / (batch_idx * loader.batch_size)
            print(
                f"[TRAIN] Epoch {epoch_num} | "
                f"Batch {batch_idx}/{len(loader)} | "
                f"Loss {loss.item():.4f} | "
                f"Avg {avg_loss_so_far:.4f} | "
                f"Elapsed {elapsed:.2f} min"
            )

    return running_loss / len(loader.dataset)


def evaluate_one_epoch(model, loader, criterion, device, epoch_num="VAL", print_summary=True):
    model.eval()

    running_loss = 0.0
    all_preds = []
    all_labels = []
    all_image_ids = []
    all_texts = []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device, non_blocking=True)
            attention_mask = batch["attention_mask"].to(device, non_blocking=True)
            pixel_values = batch["pixel_values"].to(device, non_blocking=True)
            labels = batch["labels"].to(device, non_blocking=True)

            with autocast(enabled=(device == "cuda")):
                logits = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    pixel_values=pixel_values
                )
                loss = criterion(logits, labels)

            preds = torch.argmax(logits, dim=1)

            running_loss += loss.item() * labels.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_image_ids.extend(batch["image_id"])
            all_texts.extend(batch["text"])

    avg_loss = running_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    precision_w = precision_score(all_labels, all_preds, average="weighted", zero_division=0)
    recall_w = recall_score(all_labels, all_preds, average="weighted", zero_division=0)
    f1_w = f1_score(all_labels, all_preds, average="weighted", zero_division=0)
    f1_m = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    score = compute_selection_score(f1_w, f1_m)

    p, r, f, s = precision_recall_fscore_support(
        all_labels,
        all_preds,
        labels=list(range(NUM_CLASSES)),
        zero_division=0
    )

    per_class_metrics = {
        cls: {
            "precision": float(p[i]),
            "recall": float(r[i]),
            "f1": float(f[i]),
            "support": int(s[i])
        }
        for i, cls in enumerate(range(NUM_CLASSES))
    }

    pred_df = pd.DataFrame({
        "image_id": all_image_ids,
        "text": all_texts,
        "true_label": all_labels,
        "pred_label": all_preds
    })

    if print_summary:
        print(
            f"[VAL] Epoch {epoch_num} | "
            f"Loss {avg_loss:.4f} | "
            f"Acc {acc:.4f} | "
            f"Weighted F1 {f1_w:.4f} | "
            f"Macro F1 {f1_m:.4f} | "
            f"Score {score:.4f}"
        )
        if 3 in per_class_metrics and 4 in per_class_metrics:
            print(
                f"[VAL] Class 3 -> Recall {per_class_metrics[3]['recall']:.4f}, F1 {per_class_metrics[3]['f1']:.4f} | "
                f"Class 4 -> Recall {per_class_metrics[4]['recall']:.4f}, F1 {per_class_metrics[4]['f1']:.4f}"
            )

    return {
        "loss": avg_loss,
        "acc": acc,
        "precision_weighted": precision_w,
        "recall_weighted": recall_w,
        "f1_weighted": f1_w,
        "f1_macro": f1_m,
        "score": score,
        "per_class_metrics": per_class_metrics,
        "pred_df": pred_df,
        "all_labels": all_labels,
        "all_preds": all_preds
    }

In [ ]:
labels = df_train[LABEL_COL].to_numpy()
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(labels),
    y=labels
)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

print("Class weights:", class_weights)

## 5) Train + checkpoint + evaluate

During training we:
- evaluate on the validation set each epoch
- save an epoch checkpoint (for resuming/debugging)
- save a separate **best** checkpoint using the selection score

After training we reload the best checkpoint and compute the final test metrics.


In [ ]:
config = {
    "model_version": "bert_vit_v2_2",
    "bert_name": BERT_NAME,
    "vit_name": VIT_NAME,
    "max_len": MAX_LEN,
    "batch_size": BATCH_SIZE,
    "num_classes": NUM_CLASSES,
    "dropout": DROPOUT,
    "optimizer": "AdamW",
    "head_lr": HEAD_LR,
    "backbone_lr": BACKBONE_LR,
    "weight_decay": WEIGHT_DECAY,
    "label_smoothing": LABEL_SMOOTHING,
    "use_local_image_cache": USE_LOCAL_IMAGE_CACHE,
    "active_image_dir": ACTIVE_IMAGE_DIR,
    "num_workers": NUM_WORKERS,
    "scheduler_mode": "max",
    "scheduler_target": "0.7*weighted_f1 + 0.3*macro_f1",
    "early_stopping_patience": EARLY_STOPPING_PATIENCE
}

with open(CONFIG_PATH, "w") as f:
    json.dump(config, f, indent=2)

print("Saved config to:", CONFIG_PATH)

In [ ]:
early_stopping = EarlyStoppingByScore(
    patience=EARLY_STOPPING_PATIENCE,
    delta=EARLY_STOPPING_DELTA,
    verbose=True
)

best_score = -1.0
overall_start = time.time()

for epoch in range(NUM_EPOCHS):
    print("\n" + "=" * 90)
    print(f"Starting Epoch {epoch+1}/{NUM_EPOCHS}")
    print("=" * 90)

    train_loss = train_one_epoch(
        model, train_loader, criterion, optimizer, scaler, device,
        epoch_num=epoch+1, print_every=80
    )

    val_out = evaluate_one_epoch(
        model, val_loader, criterion, device,
        epoch_num=epoch+1, print_summary=True
    )

    scheduler.step(val_out["score"])

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_out["loss"])
    history["val_acc"].append(val_out["acc"])
    history["val_precision_weighted"].append(val_out["precision_weighted"])
    history["val_recall_weighted"].append(val_out["recall_weighted"])
    history["val_f1_weighted"].append(val_out["f1_weighted"])
    history["val_f1_macro"].append(val_out["f1_macro"])
    history["val_score"].append(val_out["score"])
    history["val_cls3_recall"].append(val_out["per_class_metrics"].get(3, {}).get("recall", 0.0))
    history["val_cls3_f1"].append(val_out["per_class_metrics"].get(3, {}).get("f1", 0.0))
    history["val_cls4_recall"].append(val_out["per_class_metrics"].get(4, {}).get("recall", 0.0))
    history["val_cls4_f1"].append(val_out["per_class_metrics"].get(4, {}).get("f1", 0.0))

    val_out["pred_df"].to_csv(VAL_PRED_PATH, index=False)

    epoch_ckpt_path = os.path.join(EPOCH_CKPT_DIR, f"bert_vit_v2_2_epoch_{epoch+1}.pth")
    epoch_ckpt = {
        "epoch": epoch + 1,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "history": history,
        "config": config,
        "val_score": val_out["score"]
    }
    torch.save(epoch_ckpt, epoch_ckpt_path)
    print("Saved epoch checkpoint to:", epoch_ckpt_path)

    improved = early_stopping.step(val_out["score"])
    if improved:
        best_score = val_out["score"]
        best_ckpt = {
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
            "history": history,
            "config": config,
            "val_score": val_out["score"]
        }
        torch.save(best_ckpt, BEST_MODEL_PATH)
        print("Saved best checkpoint to:", BEST_MODEL_PATH)

    with open(HISTORY_PATH, "w") as f:
        json.dump(history, f, indent=2)
    print("Saved history to:", HISTORY_PATH)

    current_lrs = [pg["lr"] for pg in optimizer.param_groups]
    total_elapsed = (time.time() - overall_start) / 60

    print(
        f"[EPOCH END] {epoch+1}/{NUM_EPOCHS} | "
        f"Train Loss {train_loss:.4f} | "
        f"Val Loss {val_out['loss']:.4f} | "
        f"Val Acc {val_out['acc']:.4f} | "
        f"Val Weighted F1 {val_out['f1_weighted']:.4f} | "
        f"Val Macro F1 {val_out['f1_macro']:.4f} | "
        f"Val Score {val_out['score']:.4f} | "
        f"LRs {current_lrs} | "
        f"Total Elapsed {total_elapsed:.2f} min"
    )

    if early_stopping.early_stop:
        print("Early stopping triggered.")
        break

final_ckpt = {
    "epoch": len(history["train_loss"]),
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "scaler_state_dict": scaler.state_dict(),
    "history": history,
    "config": config,
    "val_score": history["val_score"][-1] if len(history["val_score"]) > 0 else None
}
torch.save(final_ckpt, FINAL_MODEL_PATH)
print("Saved final checkpoint to:", FINAL_MODEL_PATH)
print(f"Training completed in {(time.time() - overall_start)/60:.2f} minutes")

In [ ]:
plt.figure(figsize=(15, 8))

plt.subplot(2, 2, 1)
plt.plot(history["train_loss"], label="train_loss")
plt.plot(history["val_loss"], label="val_loss")
plt.title("Loss")
plt.xlabel("Epoch")
plt.legend()

plt.subplot(2, 2, 2)
plt.plot(history["val_acc"], label="val_acc")
plt.plot(history["val_f1_weighted"], label="val_weighted_f1")
plt.plot(history["val_f1_macro"], label="val_macro_f1")
plt.plot(history["val_score"], label="val_score")
plt.title("Validation metrics")
plt.xlabel("Epoch")
plt.legend()

plt.subplot(2, 2, 3)
plt.plot(history["val_cls3_recall"], label="class3_recall")
plt.plot(history["val_cls3_f1"], label="class3_f1")
plt.title("Class 3")
plt.xlabel("Epoch")
plt.legend()

plt.subplot(2, 2, 4)
plt.plot(history["val_cls4_recall"], label="class4_recall")
plt.plot(history["val_cls4_f1"], label="class4_f1")
plt.title("Class 4")
plt.xlabel("Epoch")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
best_checkpoint = torch.load(BEST_MODEL_PATH, map_location=device)

best_model = BERTViTClassifierV22(
    num_classes=NUM_CLASSES,
    bert_name=BERT_NAME,
    vit_name=VIT_NAME,
    dropout=DROPOUT
).to(device)

best_model.load_state_dict(best_checkpoint["model_state_dict"])
best_model.eval()

print("Loaded best checkpoint from epoch:", best_checkpoint["epoch"])
print("Best validation score:", best_checkpoint["val_score"])

In [ ]:
test_out = evaluate_one_epoch(
    best_model,
    test_loader,
    criterion,
    device,
    epoch_num="TEST",
    print_summary=True
)

test_out["pred_df"].to_csv(TEST_PRED_PATH, index=False)

report = classification_report(
    test_out["all_labels"],
    test_out["all_preds"],
    digits=4,
    zero_division=0
)

cm = confusion_matrix(test_out["all_labels"], test_out["all_preds"])

print("\nTest Loss:", round(test_out["loss"], 4))
print("Test Accuracy:", round(test_out["acc"], 4))
print("Test Precision (weighted):", round(test_out["precision_weighted"], 4))
print("Test Recall (weighted):", round(test_out["recall_weighted"], 4))
print("Test Weighted F1:", round(test_out["f1_weighted"], 4))
print("Test Macro F1:", round(test_out["f1_macro"], 4))

print("\nClassification Report:\n")
print(report)

print("\nConfusion Matrix:\n", cm)

with open(REPORT_PATH, "w") as f:
    f.write("BERT + ViT V2.2 Test Report\n")
    f.write("=" * 70 + "\n")
    f.write(f"Test Loss: {test_out['loss']:.4f}\n")
    f.write(f"Test Accuracy: {test_out['acc']:.4f}\n")
    f.write(f"Test Precision (weighted): {test_out['precision_weighted']:.4f}\n")
    f.write(f"Test Recall (weighted): {test_out['recall_weighted']:.4f}\n")
    f.write(f"Test Weighted F1: {test_out['f1_weighted']:.4f}\n")
    f.write(f"Test Macro F1: {test_out['f1_macro']:.4f}\n\n")
    f.write("Classification Report:\n")
    f.write(report)
    f.write("\n\nConfusion Matrix:\n")
    f.write(np.array2string(cm))

print("Saved predictions to:", TEST_PRED_PATH)
print("Saved report to:", REPORT_PATH)
print("Saved config to:", CONFIG_PATH)
print("Saved best checkpoint to:", BEST_MODEL_PATH)
print("Saved final checkpoint to:", FINAL_MODEL_PATH)